# 🎮 Game Analytics Engine — GABARITO COMENTADO

> **Uso interno do professor.** Cada célula de código contém comentários pedagógicos
> explicando o *porquê* de cada decisão, armadilhas comuns e o que observar no aluno.

---

## 📦 Imports

---
### 🗒️ Nota para o professor
O aluno provavelmente vai esquecer de importar `datetime` ou vai importar errado (`from datetime import datetime` vs `import datetime`).  
Deixe errar primeiro — o traceback é parte do aprendizado. Depois mostre a diferença:  
- `import datetime` → precisa de `datetime.datetime.now()`  
- `from datetime import datetime` → usa direto `datetime.now()` ✅ mais limpo

In [ ]:
import time
import numpy as np
from datetime import datetime

# ✅ DECISÃO: importar datetime assim permite usar datetime.now() diretamente.
# Alternativa errada comum do aluno: 'import datetime' e depois esquecer o prefixo.

---
## 1. Variáveis — Ficha de Jogador

---
### 🗒️ Nota para o professor
**O que observar no aluno:**
- Está usando nomes descritivos (`nivel` em vez de `n`)? → bom sinal
- Usou `_` como separador de milhar no `xp_atual`? → ponto extra de atenção
- Entende a diferença entre `0.68` (float) e `68` (int) para representar porcentagem?

**Pergunta provocadora para fazer em aula:**  
*"Por que `hp_percentual` é float e não int? O que muda na hora de calcular?"*  
Resposta esperada: float permite representar 68.5%, int truncaria para 68.

In [ ]:
player_id       = "PLR-00421"        # str  — ID único no banco
username        = "ShadowByte_99"    # str  — nick exibido no jogo
nivel           = 47                 # int  — nível (número inteiro, nunca 47.5)
xp_atual        = 83_450             # int  — underscore é apenas visual, Python ignora
                                     #        83_450 == 83450, mas é mais legível
hp_percentual   = 0.68               # float — representa 68% de vida
                                     #         range 0.0 a 1.0, não 0 a 100
esta_vivo       = True               # bool — exatamente dois estados: vivo ou morto
regiao          = "SA-EAST-1"        # str  — padrão de nomenclatura de servidores cloud
ping_ms         = 24                 # int  — latência em ms (sempre inteiro na prática)

print("=" * 40)
print("  FICHA DE JOGADOR — SESSÃO ATIVA")
print("=" * 40)
print(f"  ID       : {player_id}")
print(f"  Username : {username}")
print(f"  Nível    : {nivel}")
print(f"  XP       : {xp_atual:,}")           # :,  → formata com separador de milhar
                                               # 83450 vira 83,450
print(f"  HP       : {hp_percentual * 100:.0f}%")  # :.0f → sem casas decimais
print(f"  Vivo     : {esta_vivo}")
print(f"  Região   : {regiao}")
print(f"  Ping     : {ping_ms}ms")
print("=" * 40)

# ✅ DESTAQUE PEDAGÓGICO: f-string com formatação
# {xp_atual:,}          → adiciona separador de milhar
# {hp_percentual*100:.0f} → multiplica antes de formatar, arredonda p/ inteiro
# São formatações de EXIBIÇÃO — não alteram o valor da variável!

print("\n[DEBUG] Tipos:")
for var, val in [("player_id", player_id), ("nivel", nivel),
                 ("hp_percentual", hp_percentual), ("esta_vivo", esta_vivo)]:
    print(f"  {var:15} → {type(val).__name__}")

# ✅ type(val).__name__ → retorna só o nome do tipo como string ('str', 'int', etc.)
# Sem o .__name__ retornaria <class 'str'> — mais feio para exibição

---
## 2. Funções — KDA, Win Rate e Classificação

---
### 🗒️ Nota para o professor
**Armadilha clássica:** o aluno vai escrever `deaths` no denominador sem tratar o caso `deaths == 0`.  
Deixe ele testar com `deaths=0` e ver o `ZeroDivisionError`. Depois mostre o `max(1, deaths)`.

**Por que `max(1, deaths)` e não `if deaths == 0: return 0`?**  
Porque no KDA real, 0 mortes é o melhor cenário possível — a métrica não deve ser zerada,  
deve ser maximizada. `max(1, deaths)` garante divisor mínimo de 1 sem distorcer o cálculo.

**Sobre type hints (`kills: int`) e docstrings:**  
Mostre que Python não vai reclamar se o aluno passar float — as hints são para o programador, não para o interpretador. Mas em IDEs modernas (VS Code, PyCharm), elas ativam autocompletar e warnings.

In [ ]:
def calcular_kda(kills: int, deaths: int, assists: int) -> float:
    """
    Calcula o KDA ratio de um jogador.
    KDA = (kills + assists) / max(1, deaths)

    O max(1, deaths) evita divisão por zero quando o jogador não morreu.
    Um jogador com 0 mortes teria KDA = (kills + assists) / 1 — não infinito.
    """
    # ✅ PADRÃO: nunca dividir por uma variável sem garantir que não é zero
    # max(1, deaths) → se deaths=0, usa 1; se deaths>0, usa deaths
    return round((kills + assists) / max(1, deaths), 2)


def calcular_winrate(vitorias: int, total_partidas: int) -> float:
    """
    Retorna o win rate em percentual (0.0 a 100.0).
    Guard clause para jogador sem partidas — evita divisão por zero.
    """
    # ✅ PADRÃO: 'guard clause' — trata o caso inválido primeiro e sai cedo
    # Evita o if/else aninhado: código principal fica no nível raiz da função
    if total_partidas == 0:
        return 0.0
    return round((vitorias / total_partidas) * 100, 1)


def classificar_jogador(kda: float, winrate: float) -> str:
    """
    Classifica o jogador combinando KDA e win rate em um score ponderado.
    KDA tem peso 60%, win rate tem peso 40%.
    """
    # ✅ DECISÃO DE DESIGN: combinar duas métricas em um score único
    # KDA vai de ~0 a ~10+, winrate vai de 0 a 100
    # Normalizamos winrate: winrate/100 * 10 → coloca na mesma escala do KDA
    score = (kda * 0.6) + (winrate / 100 * 10 * 0.4)

    # ✅ if/elif em cascata: Python avalia de cima para baixo e para no primeiro True
    # A ordem importa! Começa do maior para o menor.
    if score >= 8:   return "💎 Diamond"
    elif score >= 6: return "🥇 Platinum"
    elif score >= 4: return "⭐ Gold"
    elif score >= 2: return "🥈 Silver"
    else:            return "🥉 Bronze"
    # ✅ O 'else' final captura tudo abaixo de 2 — não precisa de condição


jogadores_teste = [
    ("ShadowByte_99", 340, 87, 210, 178, 220),
    ("NoviceGamer01",  45, 180, 30,  12, 100),
    ("ProSniper_X",   520, 60, 180, 290, 330),
]

# ✅ f-string com alinhamento: {nome:<18} → alinha à esquerda em 18 chars
#                               {valor:>6} → alinha à direita em 6 chars
print(f"{'JOGADOR':<18} {'KDA':>6} {'WINRATE':>9} {'TIER'}")
print("-" * 48)
for nick, k, d, a, w, t in jogadores_teste:
    # ✅ Desempacotamento de tupla direto no 'for' — mais limpo que jogadores_teste[0][1]
    kda  = calcular_kda(k, d, a)
    wr   = calcular_winrate(w, t)
    tier = classificar_jogador(kda, wr)
    print(f"{nick:<18} {kda:>6.2f} {wr:>8.1f}%  {tier}")

---
## 3. `*args` e Tuplas — Processamento de Eventos de Dano

---
### 🗒️ Nota para o professor
**Conceito-chave a reforçar:** `*args` é uma tupla — imutável por design.  
Pergunte ao aluno: *"Por que faz sentido ser imutável aqui?"*  
Resposta: eventos de dano já aconteceram. Não faz sentido alterar o histórico de hits.

**Armadilha:** `sum()` e `len()` funcionam em qualquer iterável — lista, tupla, set.  
O aluno pode tentar converter para lista primeiro (`list(eventos_dano)`). Mostre que não precisa.

**Pergunta avançada:** *"O que acontece se chamar `processar_dano('PLR-001')` sem nenhum hit?"*  
Vai dar `ZeroDivisionError` no `len(eventos_dano)`. Como resolver? → guard clause no início.

In [ ]:
def processar_dano(player_id: str, *eventos_dano):
    """
    Processa todos os eventos de dano causados por um jogador em uma partida.
    *eventos_dano captura qualquer quantidade de ints como uma TUPLA.
    """
    # ✅ CONCEITO: *args sempre vira tupla — não lista!
    # Tuplas são imutáveis: eventos de dano já aconteceram, não devem ser alterados.
    print(f"\n[MATCH EVENT] Player: {player_id}")
    print(f"  Hits registrados : {len(eventos_dano)}")     # len() funciona em tupla
    print(f"  Dados brutos     : {eventos_dano}")          # exibe a tupla (com parênteses)

    dano_total = sum(eventos_dano)                         # sum() funciona em tupla
    dano_medio = dano_total / len(eventos_dano)
    dano_max   = max(eventos_dano)                         # max() funciona em tupla

    # ✅ Generator expression dentro de sum() — conta quantos hits >= 100
    # sum(1 for d in tupla if condição) → conta elementos que satisfazem a condição
    # Alternativa menos elegante: fazer um for loop com contador
    headshots = sum(1 for d in eventos_dano if d >= 100)

    print(f"  Dano total       : {dano_total}")
    print(f"  Dano médio/hit   : {dano_medio:.1f}")
    print(f"  Maior hit        : {dano_max}")
    print(f"  Headshots        : {headshots}")

    # ✅ Retorna dict — não a tupla de hits brutos
    # Quem chama a função quer o RESULTADO processado, não os dados crus
    return {
        "player": player_id,
        "total":  dano_total,
        "medio":  round(dano_medio, 1),
        "max":    dano_max,
        "hs":     headshots
    }


# ✅ Chamando com quantidades DIFERENTES de argumentos — mesma função!
# É exatamente isso que torna *args poderoso: interface estável, quantidade flexível
r1 = processar_dano("PLR-00421", 45, 120, 80, 33, 150, 95, 110, 62, 88)    # 9 hits
r2 = processar_dano("PLR-00789", 30, 75, 140, 55)                           # 4 hits
r3 = processar_dano("PLR-00112", 200, 180, 175, 190, 210, 165, 195, 188, 201, 177, 185, 193)  # 12 hits

---
## 4. `**kwargs` e Dicionários — Registro de Partidas

---
### 🗒️ Nota para o professor
**Conceito mais difícil desta seção:** o `**config` dentro do dict literal.
```python
partida = {
    "match_id": match_id,
    "modo":     modo,
    **config       ← desempacota o dict kwargs dentro do novo dict
}
```
O aluno provavelmente não vai conhecer isso. Mostre que é equivalente a `partida.update(config)` depois — mas bem mais pythônico.

**Pergunta para o aluno:** *"O que acontece se eu passar um campo que já existe como chave fixa?"*  
Ex: `registrar_partida('M1', 'ranked', match_id='OUTRO')` → o kwargs sobrescreve a chave fixa!  
Em produção isso seria um bug. Como prevenir? `**config` no final cede prioridade ao que já está no dict — na verdade, a ordem dos `**kwargs` importa.

In [ ]:
match_registry = []   # simula um banco de dados em memória


def registrar_partida(match_id: str, modo: str, **config):
    """
    Registra uma partida com configurações variáveis.
    **config é um DICIONÁRIO com todos os kwargs recebidos.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # ✅ PADRÃO AVANÇADO: desempacotamento de dict com **
    # {"a": 1, **{"b": 2, "c": 3}} vira {"a": 1, "b": 2, "c": 3}
    # config já é um dict (kwargs vira dict automaticamente)
    partida = {
        "match_id":  match_id,
        "modo":      modo,
        "timestamp": timestamp,
        **config          # expande todos os kwargs como campos do dict
    }

    match_registry.append(partida)

    print(f"\n[MATCH REGISTERED] {match_id} — {modo.upper()}")
    # ✅ .items() → retorna pares (chave, valor) para iterar no dict
    for chave, valor in partida.items():
        print(f"  {chave:<20} : {valor}")

    return partida


# ✅ Cada chamada passa campos COMPLETAMENTE diferentes — a função não quebra!
# Isso é a flexibilidade do **kwargs: a assinatura da função não precisa mudar
# quando novos modos de jogo são adicionados ao sistema.

registrar_partida(
    "MATCH-9981", "ranked",
    mapa="Distrito_Neon",
    jogadores=10,
    duracao_seg=1847,
    mvp="ShadowByte_99",
    temporada="S7"
)

registrar_partida(
    "MATCH-9982", "torneio",
    mapa="Bunker_Alpha",
    times=2,
    prize_pool_usd=5000,
    transmissao_ao_vivo=True,
    organizador="ESL_Brazil"
)

registrar_partida(
    "MATCH-9983", "casual",
    mapa="Praia_Caos",
    jogadores=20,
    modo_especial="gravidade_reduzida"
)

print(f"\n[REGISTRY] Total de partidas registradas: {len(match_registry)}")

---
## 5. Decoradores — Audit Log e Medição de Latência

---
### 🗒️ Nota para o professor
**Este é o conceito mais difícil do projeto.** Reserve tempo aqui.

**Fluxo mental para o aluno entender:**
```
@audit_log          ← decorador EXTERNO
@medir_latencia     ← decorador INTERNO
def aplicar_ban(...): ...
```
É equivalente a: `aplicar_ban = audit_log(medir_latencia(aplicar_ban))`

Quando você chama `aplicar_ban(...)`, na verdade chama:
1. `audit_log.wrapper` → imprime log, chama...
2. `medir_latencia.wrapper` → mede tempo, chama...
3. `aplicar_ban` original

**Pergunta para o aluno:** *"O que muda se inverter a ordem dos decoradores?"*  
Com a ordem atual, o AUDIT mede o tempo do `medir_latencia` + `aplicar_ban`.  
Invertida, o PERF mede só o `aplicar_ban`. Em produção, qual é mais útil? Depende do SLA.

**Por `*args, **kwargs` no wrapper?**  
Porque o wrapper não sabe quais argumentos a função original vai receber.  
Usar `*args, **kwargs` garante que o wrapper repassa TUDO para a função original.

In [ ]:
import time

def audit_log(func):
    """
    Decorador de auditoria: registra quem chamou, com quais args e o resultado.
    Em produção, esse log iria para um sistema como Datadog, Splunk ou CloudWatch.
    """
    # ✅ O wrapper recebe *args e **kwargs porque não sabe o que a função
    # decorada espera — ele precisa ser genérico para decorar QUALQUER função
    def wrapper(*args, **kwargs):
        ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]  # [:-3] corta os últimos 3 chars (microssegundos)
        print(f"[AUDIT {ts}] ▶ {func.__name__}() | args={args} kwargs={kwargs}")
        # ✅ func.__name__ → nome da função ORIGINAL, não do wrapper
        # Sem isso, todos os logs mostrariam 'wrapper' — inútil para debugging
        resultado = func(*args, **kwargs)   # chama a função original com os mesmos args
        print(f"[AUDIT {ts}] ✓ {func.__name__}() → {resultado}")
        return resultado
    return wrapper   # retorna o wrapper, não o resultado da função!


def medir_latencia(func):
    """
    Decorador de performance: mede o tempo de execução em milissegundos.
    Alerta visual se ultrapassar 100ms (SLA típico de servidor de jogo).
    """
    def wrapper(*args, **kwargs):
        # ✅ perf_counter() é mais preciso que time() para medir intervalos curtos
        # time() mede tempo do sistema (pode pular); perf_counter() é monotônico
        inicio    = time.perf_counter()
        resultado = func(*args, **kwargs)
        latencia  = (time.perf_counter() - inicio) * 1000   # converte para ms

        # ✅ Operador ternário em Python: valor_true if condição else valor_false
        status = "✅ OK" if latencia < 100 else "⚠️  SLOW"
        print(f"[PERF] {func.__name__}() → {latencia:.2f}ms {status}")
        return resultado
    return wrapper


# ✅ STACKING de decoradores: lidos de baixo para cima na aplicação,
# executados de cima para baixo na chamada.
# @audit_log vai 'por fora' de @medir_latencia.

@audit_log
@medir_latencia
def aplicar_ban(player_id: str, motivo: str, duracao_horas: int = 24):
    """Aplica ban em um jogador. Operação crítica — precisa de audit trail completo."""
    time.sleep(0.002)   # simula operação no banco de dados
    return {"status": "BANNED", "player": player_id, "horas": duracao_horas}


@audit_log
@medir_latencia
def atualizar_ranking(player_id: str, pontos_delta: int):
    """Atualiza MMR do jogador. Leitura no Redis, escrita no PostgreSQL."""
    time.sleep(0.001)   # simula escrita no Redis
    novo_mmr = 1850 + pontos_delta
    return {"player": player_id, "novo_mmr": novo_mmr}


print("=" * 55)
print(" SIMULANDO OPERAÇÕES PÓS-PARTIDA")
print("=" * 55)

atualizar_ranking("PLR-00421", +25)   # vitória: +25 MMR
print()
atualizar_ranking("PLR-00789", -18)   # derrota: -18 MMR
print()
aplicar_ban("PLR-09999", motivo="cheating_aimbot", duracao_horas=720)

---
## 6. Cache — Player Profile Service (Cache-Aside Pattern)

---
### 🗒️ Nota para o professor
**Conectar com o mercado:** este padrão exato (Cache-Aside com Redis) é o que Riot Games,  
Blizzard e Epic Games usam. O dicionário aqui é uma simplificação do Redis.

**Conceito de `global`:** a função usa `global _cache_hits, _db_hits`.  
Mostre que sem `global`, Python cria uma variável LOCAL e o contador externo não é modificado.  
Em código de produção, esse padrão seria uma classe com estado — mas `global` serve para  
ensinar o conceito de escopo de variáveis.

**`dict.get()` vs `dict[]`:**
- `_database["PLR-99999"]` → lança `KeyError` se não existir ❌
- `_database.get("PLR-99999")` → retorna `None` se não existir ✅
- `_database.get("PLR-99999", {})` → retorna `{}` como fallback ✅✅

In [ ]:
# Simula banco de dados (leitura lenta — 50ms de latência)
_database = {
    "PLR-00421": {"username": "ShadowByte_99",  "nivel": 47, "rank": "Platinum", "pais": "BR"},
    "PLR-00789": {"username": "DesertEagle_42",  "nivel": 31, "rank": "Gold",     "pais": "AR"},
    "PLR-00112": {"username": "ProSniper_X",     "nivel": 82, "rank": "Diamond",  "pais": "BR"},
    "PLR-00055": {"username": "CasualPlayer_77", "nivel": 12, "rank": "Bronze",   "pais": "CL"},
    "PLR-00300": {"username": "TheCommander",    "nivel": 61, "rank": "Platinum", "pais": "BR"},
}

_cache_perfis   = {}     # cache em memória (em produção seria Redis)
_access_counter = {}     # quantas vezes cada perfil foi acessado
_cache_hits     = 0      # ← variáveis de estado FORA da função
_db_hits        = 0      #   precisam de 'global' para serem modificadas dentro


def buscar_perfil(player_id: str) -> dict | None:
    """
    Cache-Aside Pattern:
    1. Verifica o cache primeiro
    2. Se não estiver no cache, busca no banco
    3. Popula o cache com o resultado do banco
    """
    global _cache_hits, _db_hits
    # ✅ 'global' permite modificar variáveis do escopo do módulo
    # Sem 'global', _cache_hits += 1 criaria uma variável LOCAL e a externa ficaria 0

    # ✅ dict.get(key, default) — mais seguro que dict[key]
    # Se player_id não estiver no dict, retorna 0 (não lança KeyError)
    _access_counter[player_id] = _access_counter.get(player_id, 0) + 1

    # ── CACHE HIT ─────────────────────────────────────────────────────────────
    if player_id in _cache_perfis:
        _cache_hits += 1
        print(f"  [CACHE HIT]  {player_id} → {_cache_perfis[player_id]['username']}")
        return _cache_perfis[player_id]   # retorna imediatamente — sem tocar no banco!

    # ── CACHE MISS → vai ao banco ──────────────────────────────────────────────
    _db_hits += 1
    time.sleep(0.05)   # simula latência real de query no banco (~50ms)
    perfil = _database.get(player_id)   # None se não existir

    if perfil:
        _cache_perfis[player_id] = perfil   # ✅ POPULA O CACHE para próximas buscas
        print(f"  [DB QUERY]   {player_id} → {perfil['username']} (cacheado)")
    else:
        print(f"  [NOT FOUND]  {player_id}")

    return perfil


# Simulando lobby: jogadores consultam perfis uns dos outros
requisicoes = [
    "PLR-00421", "PLR-00789", "PLR-00421",
    "PLR-00112", "PLR-00055", "PLR-00421",
    "PLR-00300", "PLR-00789", "PLR-00999",   # PLR-999 não existe no banco
    "PLR-00112", "PLR-00300", "PLR-00112",
]

print("=" * 50)
print(" LOBBY — REQUISIÇÕES DE PERFIL")
print("=" * 50)

for pid in requisicoes:
    buscar_perfil(pid)

# ✅ Relatório de eficiência do cache
total_req = len(requisicoes)
print(f"\n{'=':=<50}")
print(f"  Total requisições : {total_req}")
print(f"  DB queries        : {_db_hits}  ({_db_hits/total_req*100:.0f}%)")
print(f"  Cache hits        : {_cache_hits}  ({_cache_hits/total_req*100:.0f}%)")

# ✅ sorted() com key=lambda: ordena por valor, não por chave
# key=lambda x: -x[1] → ordena decrescente pelo segundo elemento do par (count)
print(f"\n  Acessos por jogador:")
for pid, count in sorted(_access_counter.items(), key=lambda x: -x[1]):
    username = _database.get(pid, {}).get('username', 'desconhecido')
    print(f"    {username:<22} : {count}x")

---
## 7. Exceptions — Validação da In-Game Store

---
### 🗒️ Nota para o professor
**Por que exceptions customizadas em vez de `if/else`?**  
Com `if/else`, o erro é silenciado e o código continua. Com `raise`, o fluxo para imediatamente  
e o erro sobe a pilha de chamadas — garantindo que nenhum cheque seja pulado acidentalmente.

**Sobre o `finally`:** o aluno vai achar estranho o `finally` rodar até em caso de erro.  
Mostre que é exatamente isso que o torna útil: garante que o log de auditoria SEMPRE acontece,  
independente do resultado da transação. Em bancos de dados, é onde você faz `connection.close()`.

**Armadilha do `except` genérico:**
```python
except Exception as e:   # ← pega TUDO, incluindo bugs que deveriam aparecer!
```
Mostre que capturar específico é sempre melhor — você não quer esconder um `TypeError`  
acidental atrás de uma mensagem de `SaldoInsuficienteError`.

In [ ]:
# ✅ PADRÃO: exceptions customizadas herdam de Exception
# Cada erro tem um nome que descreve EXATAMENTE o que aconteceu
# 'pass' porque não precisam de comportamento extra — só do nome
class SaldoInsuficienteError(Exception):     pass
class ItemNaoEncontradoError(Exception):     pass
class JogadorBanidoError(Exception):         pass
class LimiteTransacaoDiariaError(Exception): pass


_catalogo = {
    "skin_dragon":   {"nome": "Skin Dragão Ancestral",  "preco": 1500},
    "passe_batalha": {"nome": "Passe de Batalha S7",    "preco": 950},
    "emote_dance":   {"nome": "Emote: Victory Dance",   "preco": 300},
    "arma_gold":     {"nome": "Skin Arma Gold Edition", "preco": 2200},
}

_jogadores_banidos = {"PLR-09999"}   # ✅ set — busca O(1), mais eficiente que lista


def comprar_item(player_id: str, item_id: str, saldo_gc: int,
                 gasto_hoje: int = 0, limite_diario: int = 5000) -> dict:
    """
    Processa compra na In-Game Store.
    Validações em ordem de custo: ban (barato) → item (dict lookup) → limites numéricos.
    """
    try:
        # ✅ 'in' em set é O(1) — não percorre elemento por elemento
        if player_id in _jogadores_banidos:
            raise JogadorBanidoError(
                f"Player {player_id} está banido e não pode realizar compras.")

        # ✅ 'not in' em dict verifica as CHAVES — semântica clara
        if item_id not in _catalogo:
            raise ItemNaoEncontradoError(
                f"Item '{item_id}' não encontrado no catálogo.")

        item  = _catalogo[item_id]    # seguro: já validamos que a chave existe
        preco = item["preco"]

        if gasto_hoje + preco > limite_diario:
            raise LimiteTransacaoDiariaError(
                f"Limite diário de {limite_diario} GC atingido. "
                f"Gasto hoje: {gasto_hoje} GC | Tentativa: {preco} GC")

        if saldo_gc < preco:
            raise SaldoInsuficienteError(
                f"Saldo insuficiente: {saldo_gc} GC disponível, {preco} GC necessário.")

        # Chega aqui só se TODAS as validações passaram
        novo_saldo = saldo_gc - preco
        resultado  = {
            "status":     "SUCCESS",
            "player":     player_id,
            "item":       item["nome"],
            "pago":       preco,
            "novo_saldo": novo_saldo
        }
        print(f"  ✅ COMPRA APROVADA: {item['nome']} por {preco} GC | Saldo: {novo_saldo} GC")
        return resultado

    # ✅ PADRÃO: capturar do mais específico para o mais genérico
    # Cada except trata um erro diferente com uma mensagem diferente
    except JogadorBanidoError         as e: print(f"  🚫 BANIDO          : {e}")
    except ItemNaoEncontradoError     as e: print(f"  ❌ ITEM INVÁLIDO   : {e}")
    except LimiteTransacaoDiariaError as e: print(f"  ⛔ LIMITE DIÁRIO  : {e}")
    except SaldoInsuficienteError     as e: print(f"  💸 SALDO INSUF.   : {e}")
    finally:
        # ✅ 'finally' SEMPRE executa — com sucesso ou com erro
        # Ideal para logs de auditoria, fechar conexões, liberar recursos
        print(f"  [LOG] Tentativa de compra registrada — player={player_id}, item={item_id}")


print("=" * 60)
print(" IN-GAME STORE — PROCESSANDO TRANSAÇÕES")
print("=" * 60)

print("\n[TX-001] Compra válida:")
comprar_item("PLR-00421", "passe_batalha", saldo_gc=2000)

print("\n[TX-002] Saldo insuficiente:")
comprar_item("PLR-00789", "arma_gold", saldo_gc=500)

print("\n[TX-003] Item não existe:")
comprar_item("PLR-00112", "item_inexistente_xp", saldo_gc=9999)

print("\n[TX-004] Jogador banido tentando comprar:")
comprar_item("PLR-09999", "emote_dance", saldo_gc=5000)

print("\n[TX-005] Limite diário atingido:")
comprar_item("PLR-00300", "skin_dragon", saldo_gc=9999, gasto_hoje=4800)

---
## 8. NumPy — Season Report e Matchmaking Analytics

---
### 🗒️ Nota para o professor
**Por que NumPy e não listas?**  
Demonstre com timing: `sum()` em lista de 1M elementos vs `np.array.sum()`.  
NumPy opera em C por baixo — 10-100x mais rápido para operações numéricas.

**Conceito mais difícil: indexação booleana**
```python
kills_em_vitorias = kills[resultado == 1]
```
Mostre passo a passo:
1. `resultado == 1` → array de True/False
2. `kills[array_bool]` → filtra só onde é True

Isso é **vetorização** — sem loop explícito. Em dados de 10 milhões de partidas, a diferença é abissal.

**Correlação:** explique que 0.99 significa que kills e dano são quase perfeitamente correlacionados — o que faz sentido no contexto. Esse tipo de análise é base do matchmaking por MMR.

In [ ]:
import numpy as np

# ✅ np.array() converte uma lista Python em array NumPy
# A diferença: arrays NumPy têm operações vetorizadas; listas não
kills_por_partida  = np.array([8, 12, 3, 15, 7, 20, 5, 11, 9, 18,
                                6, 14, 2, 16, 10, 13, 4, 19, 8, 11,
                               17, 6, 12, 3, 14, 9, 21, 7, 15, 10])

dano_por_partida   = np.array([820, 1240, 310, 1580, 740, 2100, 490, 1120, 940, 1870,
                                630, 1450, 210, 1690, 1050, 1380, 420, 1950, 830, 1180,
                               1780, 610, 1290, 340, 1470, 980, 2250, 710, 1560, 1080])

# ✅ Resultado como 0/1 permite usar como máscara booleana e também somar (sum = total de vitórias)
resultado_partida  = np.array([1, 1, 0, 1, 0, 1, 0, 1, 1, 1,
                                0, 1, 0, 1, 1, 1, 0, 1, 0, 0,
                                1, 0, 1, 0, 1, 1, 1, 0, 1, 1])

print("=" * 55)
print("  📊 SEASON REPORT — ShadowByte_99")
print("=" * 55)

# ✅ Métodos nativos do array — sem precisar de loop:
# .sum() .mean() .max() .min() .std()
print("\n[KILLS]")
print(f"  Total    : {kills_por_partida.sum()}")
print(f"  Média    : {kills_por_partida.mean():.1f} por partida")
print(f"  Melhor   : {kills_por_partida.max()} kills")
print(f"  Pior     : {kills_por_partida.min()} kills")
print(f"  Desvio   : {kills_por_partida.std():.1f}  ← quanto o jogador oscila entre partidas")
# ✅ Desvio padrão baixo = consistente. Alto = irregular (bom às vezes, ruim outras)

print("\n[DANO]")
print(f"  Total    : {dano_por_partida.sum():,} de dano na temporada")
print(f"  Média    : {dano_por_partida.mean():.0f} por partida")
print(f"  Pico     : {dano_por_partida.max():,}")

print("\n[WIN RATE]")
# ✅ resultado_partida.sum() funciona porque 1+1+0+1 = total de vitórias
total_vitorias = resultado_partida.sum()
total_partidas = len(resultado_partida)
winrate = (total_vitorias / total_partidas) * 100
print(f"  Vitórias : {total_vitorias}/{total_partidas}")
print(f"  Win Rate : {winrate:.1f}%")

# ── INDEXAÇÃO BOOLEANA — o poder real do NumPy ────────────────────────────────
print("\n[ANÁLISE AVANÇADA]")

# ✅ kills_por_partida >= 15 → array de True/False
# kills_por_partida[array_bool] → retorna só os elementos onde é True
partidas_carry = kills_por_partida[kills_por_partida >= 15]
partidas_ruim  = kills_por_partida[kills_por_partida <= 5]

# ✅ Filtra kills apenas nas partidas onde resultado == 1 (vitória)
# Dois arrays sincronizados: mesmo índice = mesma partida
partidas_vitoria_kills = kills_por_partida[resultado_partida == 1]
partidas_derrota_kills = kills_por_partida[resultado_partida == 0]

print(f"  Partidas carry (≥15 kills)  : {len(partidas_carry)} partidas")
print(f"  Partidas ruins (≤5 kills)   : {len(partidas_ruim)} partidas")
print(f"  Média kills em vitórias     : {partidas_vitoria_kills.mean():.1f}")
print(f"  Média kills em derrotas     : {partidas_derrota_kills.mean():.1f}")

# ── PERCENTIS — usado em matchmaking para separar tiers ──────────────────────
print("\n[MATCHMAKING STATS — PERCENTIS DE DANO]")
# ✅ np.percentile(array, p): p=50 é a mediana, p=90 são os top 10% mais fortes
for p in [25, 50, 75, 90]:
    val = np.percentile(dano_por_partida, p)
    print(f"  P{p:<3} : {val:>7.0f} de dano")

# ── CORRELAÇÃO — análise de causalidade ──────────────────────────────────────
# ✅ np.corrcoef() retorna matriz 2x2 de correlação
# [0,1] é a correlação entre os dois arrays (0=sem correlação, 1=correlação perfeita)
correlacao = np.corrcoef(kills_por_partida, dano_por_partida)[0, 1]
print(f"\n  Correlação kills↔dano : {correlacao:.3f}")
print(f"  (próximo de 1.0 = mais kills → mais dano — faz sentido!)")

print("\n[AVALIAÇÃO FINAL]")
if winrate >= 60 and kills_por_partida.mean() >= 10:
    print("  🏆 Performance ELITE — Elegível para ranking Diamond")
elif winrate >= 50:
    print("  ⭐ Performance ACIMA DA MÉDIA — Mantendo Platinum")
else:
    print("  📉 Performance ABAIXO DA META — Risco de rebaixamento")

---
## 9. Pipeline Completo — Pós-Partida

---
### 🗒️ Nota para o professor
**Este é o momento de síntese.** O aluno deve reconhecer cada bloco que usou antes.

**Ponto de atenção:** `*p['hits']` na chamada `processar_dano(p['player_id'], *p['hits'])`.  
O `*` aqui **desempacota a lista** como argumentos posicionais — o oposto do que `*args` faz  
na definição. Mostre essa simetria:
- Na **definição**: `*args` AGRUPA argumentos em tupla  
- Na **chamada**: `*lista` EXPANDE a lista em argumentos separados

**Perguntas de revisão para encerrar a aula:**
1. Onde está o uso de `*args`? (processar_dano)
2. Onde está o uso de `**kwargs`? (registrar_partida, usada anteriormente)
3. Qual função tem decorador? (atualizar_ranking)
4. O que garante que o log de auditoria sempre acontece mesmo com erro? (finally)
5. Como o cache evita ir ao banco duas vezes? (checa dict antes de buscar)

In [ ]:
def pipeline_pos_partida(match_id: str, resultados: list[dict]):
    """
    Pipeline de pós-partida: integra todos os módulos construídos.
    Em produção, cada bloco seria uma chamada de serviço separado (microsserviços).
    """
    print(f"\n{'='*60}")
    print(f"  🏁 MATCH OVER — {match_id}")
    print(f"{'='*60}")

    for p in resultados:
        print(f"\n  ── {p['username']} ──")

        # ── Bloco 1: métricas (funções simples) ───────────────────────────────
        kda  = calcular_kda(p['kills'], p['deaths'], p['assists'])
        tier = classificar_jogador(kda, p['winrate_historico'])

        # ── Bloco 2: análise de dano (*args) ──────────────────────────────────
        # ✅ *p['hits'] EXPANDE a lista como argumentos separados
        # processar_dano('PLR-001', *[95, 110, 145]) ==
        # processar_dano('PLR-001', 95, 110, 145)
        stats_dano = processar_dano(p['player_id'], *p['hits'])

        # ── Bloco 3: ranking (decoradores rodam automaticamente) ───────────────
        # ✅ O aluno não chama os decoradores diretamente
        # Eles executam transparentemente em toda chamada de atualizar_ranking()
        delta = +20 if p['venceu'] else -15
        atualizar_ranking(p['player_id'], delta)

        print(f"  Tier calculada : {tier}")


# Dados simulados de uma partida finalizada
resultado_partida_final = [
    {
        "player_id": "PLR-00421", "username": "ShadowByte_99",
        "kills": 14, "deaths": 3, "assists": 6,
        "hits": [95, 110, 145, 80, 200, 75, 130, 88],
        "winrate_historico": 64.2, "venceu": True
    },
    {
        "player_id": "PLR-00789", "username": "DesertEagle_42",
        "kills": 5, "deaths": 9, "assists": 2,
        "hits": [40, 55, 30, 70, 48],
        "winrate_historico": 41.0, "venceu": False
    },
]

pipeline_pos_partida("MATCH-9981", resultado_partida_final)

print(f"\n\n{'='*60}")
print("  ✅ Pipeline concluído. Todos os dados foram gravados.")
print(f"{'='*60}")

# ── RECAPITULAÇÃO FINAL ────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════╗
║              CONCEITOS USADOS NESTE PROJETO              ║
╠══════════════════════════════════════════════════════════╣
║  Variáveis   → Ficha de jogador (str, int, float, bool)  ║
║  Funções     → KDA, Win Rate, Classificação de tier      ║
║  *args       → Processar N hits de dano por partida      ║
║  **kwargs    → Registrar partidas com campos flexíveis   ║
║  Decoradores → Audit log + medição de latência           ║
║  Cache       → Cache-Aside para perfis de jogadores      ║
║  Exceptions  → Validação de transações na loja           ║
║  NumPy       → Season Report e analytics de matchmaking  ║
╚══════════════════════════════════════════════════════════╝
""")